# Healthcare Analytics Lakehouse Pipeline - Gold Layer (Analytics & Aggregations)

This notebook demonstrates the aggregation and business intelligence process for the **Gold Layer** of a Healthcare Data Lakehouse using **PySpark** and **Delta Lake** on Databricks[cite: 4].

### Objectives
1. **Load Silver Data:** Read consolidated silver hospital records[cite: 4].
2. **Department Performance:** Compute total patients, admissions, average age, and average stay per department[cite: 4].
3. **Disease Metrics:** Aggregate admission metrics grouped by diagnosis[cite: 4].
4. **Patient Summaries:** Summarize total admissions and hospital days per patient[cite: 4].
5. **Doctor Metrics:** Analyze throughput and average stay duration per doctor[cite: 4].
6. **Hospital KPIs:** Calculate overall executive-level KPIs[cite: 4].
7. **Gold Table Persistence:** Persist all aggregated datasets as Gold Delta Tables in Unity Catalog (`workspace.healthcare`)[cite: 4].

## Step 1: Read Consolidated Silver Data
We start by loading the consolidated hospital records from the Silver Delta table `workspace.healthcare.silver_hospital_records`[cite: 4].

In [0]:
hospital_records = spark.table("workspace.healthcare.silver_hospital_records")
display(hospital_records)

## Step 2: Inspect Data Schema
We inspect the schema of `hospital_records` to verify available attributes such as patient demographics, admission details, physician information, and clinical metrics[cite: 4].

In [0]:
hospital_records.printSchema()

## Step 3: Import PySpark Functions
Import essential PySpark SQL aggregation and column transformation functions (`countDistinct`, `count`, `avg`, `round`, `col`) required for analytics[cite: 4].

In [0]:
from pyspark.sql.functions import (
    countDistinct,
    count,
    avg,
    round,
    col
)

## Step 4: Calculate Department Performance Metrics
We group data by `department` to calculate total unique patients, total unique admissions, average patient age, and average length of stay (rounded to 2 decimal places)[cite: 4].

In [0]:
department_performance = hospital_records.groupBy(
    "department"
).agg(
    countDistinct("patient_id").alias(
        "total_patients"
    ),
    
    countDistinct("admission_id").alias(
        "total_admissions"
    ),
    
    round(
        avg("age"),
        2
    ).alias(
        "average_patient_age"
    ),
    
    round(
        avg("length_of_stay"),
        2
    ).alias(
        "average_length_of_stay"
    )
)

## Step 5: Display Department Performance Data
Reviewing the aggregated department performance DataFrame[cite: 4].

In [0]:
display(department_performance)

## Step 6: Save Department Performance to Gold Layer
We write the aggregated performance metrics directly into the Gold Delta table `workspace.healthcare.gold_department_performance`[cite: 4].

In [0]:
department_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.healthcare.gold_department_performance")

## Step 7: Verify Gold Department Performance Table
Confirming that the saved Delta table returns valid records directly from storage[cite: 4].

In [0]:
display(spark.table("workspace.healthcare.gold_department_performance"))

## Step 8: Calculate Disease Summary Metrics
We aggregate records by `diagnosis` to analyze unique patients, total admissions, and average length of stay per medical condition, ordered by total admissions descending[cite: 4].

In [0]:
disease_summary = hospital_records.groupBy(
    "diagnosis"
).agg(
    countDistinct("patient_id").alias(
        "total_patients"
    ),
    
    countDistinct("admission_id").alias(
        "total_admissions"
    ),
    
    round(
        avg("length_of_stay"),
        2
    ).alias(
        "average_length_of_stay"
    )
).orderBy(
    col("total_admissions").desc()
)

## Step 9: Display Disease Summary Data
Inspecting the computed disease metrics[cite: 4].

In [0]:
display(disease_summary)

## Step 10: Save Disease Summary to Gold Layer
Writing the disease summary aggregate data into Delta table `workspace.healthcare.gold_disease_summary`[cite: 4].

In [0]:
disease_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.healthcare.gold_disease_summary")

## Step 11: Calculate Patient Level Summaries
We aggregate patient records by demographic fields to extract total individual admissions, total stay duration in days, and average length of stay[cite: 4].

In [0]:
patient_summary = hospital_records.groupBy(
    "patient_id",
    "first_name",
    "last_name",
    "gender",
    "age",
    "city"
).agg(
    countDistinct("admission_id").alias(
        "total_admissions"
    ),
    
    sum("length_of_stay").alias(
        "total_hospital_days"
    ),
    
    round(
        avg("length_of_stay"),
        2
    ).alias(
        "average_length_of_stay"
    )
)

## Step 12: Import PySpark Sum Function
Importing `sum` function from `pyspark.sql.functions` for aggregating cumulative stay durations[cite: 4].

In [0]:
from pyspark.sql.functions import sum

## Step 13: Display Patient Summary Data
Inspecting the computed per-patient statistics[cite: 4].

In [0]:
display(patient_summary)

## Step 14: Save Patient Summary to Gold Layer
Writing the aggregated patient metrics to `workspace.healthcare.gold_patient_summary`[cite: 4].

In [0]:
patient_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.healthcare.gold_patient_summary")

## Step 15: Calculate Doctor Performance Metrics
Grouping by physician attributes (`doctor_id`, `doctor_name`, `department`, `specialization`) to compute total patients treated, total admissions handled, and average stay duration[cite: 4].

In [0]:
doctor_performance = hospital_records.groupBy(
    "doctor_id",
    "doctor_name",
    "department",
    "specialization"
).agg(
    countDistinct("patient_id").alias(
        "total_patients"
    ),
    
    countDistinct("admission_id").alias(
        "total_admissions"
    ),
    
    round(
        avg("length_of_stay"),
        2
    ).alias(
        "average_length_of_stay"
    )
)

## Step 16: Display Doctor Performance Data
Reviewing physician performance results[cite: 4].

In [0]:
display(doctor_performance)

## Step 17: Save Doctor Performance to Gold Layer
Writing doctor performance analytics to `workspace.healthcare.gold_doctor_performance`[cite: 4].

In [0]:
doctor_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.healthcare.gold_doctor_performance"
    )

## Step 18: Calculate Global Hospital KPIs
Computing global executive KPIs across the entire hospital system (total distinct patients, total admissions, average patient age, and overall average stay duration)[cite: 4].

In [0]:
hospital_kpis = hospital_records.agg(
    countDistinct("patient_id").alias(
        "total_patients"
    ),
    
    countDistinct("admission_id").alias(
        "total_admissions"
    ),
    
    round(
        avg("age"),
        2
    ).alias(
        "average_patient_age"
    ),
    
    round(
        avg("length_of_stay"),
        2
    ).alias(
        "average_length_of_stay"
    )
)

## Step 19: Display Global Hospital KPIs
Inspecting the hospital KPI metrics[cite: 4].

In [0]:
display(hospital_kpis)

## Step 20: Save Hospital KPIs to Gold Layer
Persisting the global KPI table to `workspace.healthcare.gold_hospital_kpis`[cite: 4].

In [0]:
hospital_kpis.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.healthcare.gold_hospital_kpis")

## Step 21: Verify Database Architecture
List all populated tables in the `workspace.healthcare` schema to verify that Bronze, Silver, and Gold layer tables are successfully instantiated[cite: 4].

In [0]:
spark.sql("""SHOW TABLES IN workspace.healthcare""").show(truncate=False)